# Credit Card Fraud Detection using Decision Tree and SVM

This notebook contains the exploratory data analysis, preprocessing, and model implementation for credit card fraud detection.

### Baseline Models:
1. **Decision Tree Classifier**
2. **Support Vector Machine (SVM)**

### Modern Additions (FraudLens AI Production Engine):
3. **Isolation Forest (Unsupervised Anomaly Detection)**
4. **XGBoost / Gradient Boosting Classifier**

### Dataset Description:
- 284,807 transactions made by European cardholders in September 2013.
- Features `V1` to `V28` are principal components obtained with PCA for confidentiality.
- Features `Time` and `Amount` are non-transformed.
- Feature `Class` is the response variable: `0` for normal transaction, `1` for fraud (492 frauds, ~0.172%).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

import joblib
import os

## 1. Data Loading & Preprocessing
We load the dataset and apply RobustScaler to `Amount` and `Time` features to minimize outlier sensitivity.

In [2]:
# Preprocessing pipeline
def preprocess_creditcard_data(df):
    rob_scaler = RobustScaler()
    df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
    df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))
    df.drop(['Time', 'Amount'], axis=1, inplace=True)
    
    X = df.drop('Class', axis=1)
    y = df['Class']
    
    return X, y, rob_scaler

## 2. Decision Tree & SVM Training
Train Decision Tree and SVM on balanced/resampled training data.

In [3]:
# Decision Tree Baseline
dt_model = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)

# SVM Baseline
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)

# Modern Anomaly Model: Isolation Forest
iso_forest = IsolationForest(contamination=0.002, random_state=42)

## 3. Evaluation Metrics
Evaluate with ROC-AUC, PR-AUC, and Confusion Matrix to handle extreme class imbalance.